# Notebook for Exploratory Data Analysis
### For Analyzing Bias after Vision Zero

In [ ]:
# install packages if necessary
%pip install -r "../requirements.txt"

In [1]:
import pandas as pd
import geopandas as gpd
import zipfile

### Function to load data
- Loads in police stop data from the [Stanford Opening Policing Project](https://openpolicing.stanford.edu/data/)
- Data has been cleaned in the stop preprocessing notebook 

In [2]:
def load_data(path, lat = None, long = None, geospatial = False):
    # path is the string of the path to datafile
    # lat/long are strings for the name of the latitude and longitude columns in the datafile
    # set geospatial to true if trying to extract xy coordinates from a csv file
    
    # unzip data if name ends with .zip
    if(path[-3:]=="zip"):
        with zipfile.ZipFile(path) as z:
            csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
            df = pd.read_csv(z.open(csv_file))
        print("Unzipped")
    # read in regularly for non zipped files
    else:
        # split into last file name after /
        if("geojson" in path.rsplit('/', 1)[-1]):
            df = gpd.read_file(path)
        else:
            df = pd.read_csv(path)

    # convert to a geospatial dataframe to extract x,y coordinates
    if(geospatial):
        df = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[long], df[lat]),
            # crs set for lat/long data
            crs="EPSG:4326" 
        )

        # Print coordinate reference system to verify correct locations
        print("Data CRS:", df.crs)
    
    return df

In [8]:
stops_path = r"./data/stops_clean.csv.zip"
stops_df = load_data(stops_path)

Unzipped


In [9]:
# Recode NAs to 'None' for epc_class
stops_df = stops_df['epc_class'].fillna('None')